# 02 — Feature Engineering
**Fraud Detection & Anomaly Scoring System**

Goals:
- Create domain-informed features (log-amount, z-score, hour)
- Handle class imbalance with SMOTE
- Scale features for model consumption
- Analyse feature importance before modelling

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from collections import Counter

plt.rcParams.update({'figure.dpi': 120})
sns.set_theme(style='darkgrid')

from src.features import load_raw, engineer_features, build_train_test

df_raw = load_raw()
df_eng = engineer_features(df_raw)
print(f"Original shape   : {df_raw.shape}")
print(f"Engineered shape : {df_eng.shape}")
df_eng.head()

## 1. New Features — Visual Validation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Amount_Log
for cls, label, color in [(0,'Legitimate','#27ae60'), (1,'Fraud','#e74c3c')]:
    subset = df_eng[df_eng['Class']==cls]['Amount_Log']
    axes[0].hist(subset, bins=60, alpha=0.6, color=color, label=label, density=True)
axes[0].set_title('Amount_Log (log1p transformed)')
axes[0].legend()

# Amount_ZScore
for cls, label, color in [(0,'Legitimate','#27ae60'), (1,'Fraud','#e74c3c')]:
    subset = df_eng[df_eng['Class']==cls]['Amount_ZScore'].clip(-5, 5)
    axes[1].hist(subset, bins=60, alpha=0.6, color=color, label=label, density=True)
axes[1].set_title('Amount_ZScore (clipped ±5)')
axes[1].legend()

# Hour
fraud_hours = df_eng[df_eng['Class']==1]['Hour'].value_counts().sort_index()
axes[2].bar(fraud_hours.index, fraud_hours.values, color='#e74c3c', alpha=0.8)
axes[2].set_title('Fraud by Hour of Day')
axes[2].set_xlabel('Hour')

plt.suptitle('Engineered Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/fe_new_features.png', dpi=120)
plt.show()

## 2. SMOTE — Before & After

In [ ]:
from sklearn.model_selection import train_test_split

y = df_eng['Class'].values
X = df_eng.drop(columns=['Class']).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print("Before SMOTE:", Counter(y_train))

sm = SMOTE(sampling_strategy=0.1, random_state=42)
X_res, y_res = sm.fit_resample(X_train_s, y_train)
print("After SMOTE :", Counter(y_res))

# Visual
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
before = Counter(y_train)
after  = Counter(y_res)

for ax, counts, title in [
    (axes[0], before, 'Before SMOTE'),
    (axes[1], after,  'After SMOTE'),
]:
    ax.bar(['Legitimate','Fraud'], [counts[0], counts[1]],
           color=['#27ae60','#e74c3c'], edgecolor='white')
    ax.set_title(title)
    ax.set_ylabel('Count')
    for i, v in enumerate([counts[0], counts[1]]):
        ax.text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

plt.suptitle('SMOTE: Addressing Class Imbalance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/fe_smote_comparison.png', dpi=120)
plt.show()

## 3. Feature Variance (after scaling)

In [ ]:
feat_names = df_eng.drop(columns=['Class']).columns.tolist()
variances = pd.Series(X_train_s.var(axis=0), index=feat_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(variances.index, variances.values, color='#3498db', edgecolor='white')
ax.set_title('Feature Variance (scaled training set)', fontsize=13, fontweight='bold')
ax.set_ylabel('Variance')
ax.tick_params(axis='x', rotation=60)
plt.tight_layout()
plt.savefig('../reports/fe_feature_variance.png', dpi=120)
plt.show()

## 4. Quick Mutual Information Score (proxy feature importance)

In [ ]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(X_train_s, y_train, random_state=42)
mi_series = pd.Series(mi_scores, index=feat_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(mi_series.index, mi_series.values, color='#9b59b6', edgecolor='white')
ax.set_title('Mutual Information with Target (Class)', fontsize=13, fontweight='bold')
ax.set_ylabel('Mutual Information Score')
ax.tick_params(axis='x', rotation=60)
plt.tight_layout()
plt.savefig('../reports/fe_mutual_information.png', dpi=120)
plt.show()

print("\nTop 10 features by mutual information:")
print(mi_series.head(10))

## 5. Summary

| Feature | Purpose | Impact |
|---------|---------|--------|
| `Amount_Log` | Reduce right-skew | Better numerical stability |
| `Amount_ZScore` | Normalise scale | Uniform comparison |
| `Hour` | Time-of-day bucket | Captures off-hours fraud |
| SMOTE | Balance training set | Prevents majority-class bias |
| StandardScaler | Feature scaling | Required for distance-based models |

> **Next step:** Model Building (`03_modelling.ipynb`)